In [1]:
from dataloader import *
from augmentations import *
from models import *
from training import *
import matplotlib.pyplot as plt
from itertools import product
from statistics import mean, stdev
import os

/home/igor-lechoszest/anaconda3/envs/AI_Diag/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Experiment loop for hyperparameter selection

In [2]:
params = {
     #training params
    "batch_size": [8, 32, 128],
    "learning_rate": [1e-3, 1e-4, 1e-5],
    "optimizer": ['Adam', 'SGD'],
    #regularization params
    "dropout": [0.2, 0.5],
    "weight_decay": [0, 1e-3, 0.5]
}

# Create experiment grid
experiment_grid = [dict(zip(params.keys(), v)) for v in product(*params.values())]

In [3]:
# Only for testing!!!!!!
from torch.utils.data import Subset
import numpy as np

def get_subset(dataset, fraction=0.1):
    size = int(len(dataset) * fraction)
    indices = np.random.choice(len(dataset), size, replace=False)
    return Subset(dataset, indices)

In [4]:
def run_experiment(model_name,experiment_config, epoch_number=5):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model
    model = get_model(model_name)   
    model.to(device)

    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data

    transformations = basic_transforms(augmentation_type=None, model_type=model_name)

    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)
    
    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=None,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    accs = []
    f1s = []
    recalls = []
    precisions = []
    for epoch in range(epoch_number):

        train(model, train_loader, optimizer, criterion, device)
        scores = validate(model, val_loader, device, criterion)

        accs.append(scores["accuracy"])
        f1s.append(scores["f1"])
        recalls.append(scores["recall"])
        precisions.append(scores["precision"])

    val_scores = {
        'mean_acc': mean(accs),
        'std_acc': stdev(accs),

        'mean_f1': mean(f1s),
        'std_f1': stdev(f1s),

        'mean_recall': mean(recalls),
        'std_recall': stdev(recalls),
        'mean_precision': mean(precisions),
        'std_precision': stdev(precisions)
    }

    test_scores = validate(model, test_loader, device, criterion)

    return {**val_scores, **test_scores}

In [ ]:
os.environ["HSA_OVERRIDE_GFX_VERSION"] = "11.0.0"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

results = []

for i, config in enumerate(experiment_grid):
    print(config)

    model = "SmallCNN" # do wyboru jeszcze ConvNeXt, VisionTransformer

    score = run_experiment(model,config)
    d = {
        "model": model,
        "config": config}
    results.append({
        **d, **score
    })

{'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0}


/home/igor-lechoszest/anaconda3/envs/AI_Diag/lib/python3.12/site-packages/torch/nn/modules/module.py:1355: UserWarning: expandable_segments not supported on this platform (Triggered internally at /pytorch/c10/hip/HIPAllocatorConfig.h:36.)
  return t.to(


Training Loop


100%|██████████| 11250/11250 [00:25<00:00, 445.43it/s]


Validation Loop


...

100%|██████████| 11250/11250 [00:24<00:00, 450.51it/s]


Validation Loop


100%|██████████| 11250/11250 [00:20<00:00, 547.05it/s]


Training Loop


 21%|██        | 2387/11250 [00:05<00:20, 427.70it/s]

In [6]:
results

[{'model': 'SmallCNN',
  'config': {'batch_size': 8,
   'learning_rate': 0.001,
   'optimizer': 'Adam',
   'dropout': 0.2,
   'weight_decay': 0},
  'mean_acc': 0.2423,
  'std_acc': 0.01112514669066834,
  'mean_f1': 0.19518531853924737,
  'std_f1': 0.00028426325141826656,
  'mean_recall': 0.24230000000000002,
  'std_recall': 0.011125146690668358,
  'mean_precision': 0.39317422516349076,
  'std_precision': 0.04767385927913628,
  'loss': 3.691635948500741,
  'accuracy': 0.24914444444444445,
  'precision': 0.4298481127361081,
  'recall': 0.2491444444444444,
  'f1': 0.19543790074767567},
 {'model': 'SmallCNN',
  'config': {'batch_size': 8,
   'learning_rate': 0.001,
   'optimizer': 'Adam',
   'dropout': 0.2,
   'weight_decay': 0.001},
  'mean_acc': 0.2096777777777778,
  'std_acc': 0.03368970975253241,
  'mean_f1': 0.16912497742992563,
  'std_f1': 0.04057523611233439,
  'mean_recall': 0.20967777777777774,
  'std_recall': 0.03368970975253241,
  'mean_precision': 0.412362635541775,
  'std_prec

In [ ]:
#saving results to csv file
import pandas as pd
df = pd.DataFrame(results)
df.to_csv("SmallCNN_results_108_5epochs.csv", index=False)

### Tutaj wybieramy najlepsze konfiguracje (jedna na kazdy model) i zaczynamy testowanie augmentacji

## Experiment loop for auqmentation techniques

In [8]:
# przykladowo
best_conf_resnet = {'batch_size': 8, 'learning_rate': 0.001, 'optimizer': 'Adam', 'dropout': 0.2, 'weight_decay': 0.001}

In [9]:
# standard operations
basic_augmentations = ["flip", "shift", "rotation"]
# more advanced data augmentation
advanced_augmentations = 'cutmix' 

In [13]:
def run_experiment_augmentation(model_name,experiment_config,augmentation=None, cutmix_collate_fn=None, epoch_number=2):

    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

    # model
    model = get_model(model_name)   
    model.to(device)


    # optimizer
    if experiment_config["optimizer"] == "adam":
        optimizer = torch.optim.Adam(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"]
        )
    else:
        optimizer = torch.optim.SGD(
            model.parameters(),
            lr=experiment_config["learning_rate"],
            weight_decay=experiment_config["weight_decay"],
            momentum=0.9
        )

    # data
    transformations = basic_transforms(augmentation_type=augmentation, model_type=model_name)


    train_dataset = get_train_dataset(transform=transformations)
    val_dataset, test_dataset = get_val_train_dataset(model_type=model_name)
    

    train_loader = get_train_dataloaders(
        train_dataset,
        collate_fn=cutmix_collate_fn,
        batch_size=experiment_config["batch_size"]
    )
    
    val_loader, test_loader = get_val_test_dataloaders(
        val_dataset, test_dataset,
        batch_size=experiment_config["batch_size"]
    )

    criterion = torch.nn.CrossEntropyLoss()

    accs = []
    f1s = []
    recalls = []
    precisions = []
    for epoch in range(epoch_number):

        train(model, train_loader, optimizer, criterion, device)
        scores = validate(model, val_loader, device)

        accs.append(scores["accuracy"])
        f1s.append(scores["f1"])
        recalls.append(scores["recall"])
        precisions.append(scores["precision"])

    val_scores = {
        'mean_acc': mean(accs),
        'std_acc': stdev(accs),

        'mean_f1': mean(f1s),
        'std_f1': stdev(f1s),

        'mean_recall': mean(recalls),
        'std_recall': stdev(recalls),
        'mean_precision': mean(precisions),
        'std_precision': stdev(precisions)
    }

    test_scores = validate(model, test_loader, device)

    return {**val_scores, **test_scores}

In [14]:
# augmentation experiments

results = []

for augm in basic_augmentations:

    model = "ResNet"

    score = run_experiment_augmentation(model,best_conf_resnet, augm, cutmix_collate_fn=None)
    d = {
        "model": model,
        "augmentation": augm}
    
    results.append({
        **d, **score
    })

score = run_experiment_augmentation(model,best_conf_resnet, None, cutmix_collate_fn=cutmix_collate_fn)
d = {
    "model": model,
    "augmentation": 'cutmix'}
    
results.append({
        **d, **score
    })

  6%|▌         | 669/11250 [00:09<02:32, 69.55it/s]


KeyboardInterrupt: 

In [ ]:
# tutaj też sprawdzamy jak się modele zachowywały, wizualizacje i tym podobne etc.

## Few-shot learning

In [ ]:
#przyklad jak wybrać podzbior do fewshot learningu, wystarczy na wczytanym datasecie zastosować few_shot_subset

transformations = basic_transforms(augmentation_type=None, model_type='ResNet')

train_dataset = get_train_dataset(transform=transformations)
val_dataset, test_dataset = get_val_train_dataset(model_type='ResNet')

few_shot_train =  few_shot_subset(train_dataset)
few_shot_val =  few_shot_subset(val_dataset)
few_shot_test =  few_shot_subset(test_dataset)

### jak to zrobimy to w sumie już można zaczynać polerowanie wszystkiego i pisanie raportu.